# LiteLLM as an LLM Gateway — Revision Notebook

A practical walkthrough of LiteLLM's SDK, Router, and Proxy/Gateway.

## Learning goals
1. Understand the gateway architecture.
2. Configure response caching.
3. Route requests across deployments and multiple provider keys.
4. Configure retries and model fallbacks.
5. Apply rate limits and virtual-key controls.
6. Connect LangChain to LiteLLM through an OpenAI-compatible endpoint.
7. Understand operational and security trade-offs.

**Safety:** use environment variables or a secrets manager for provider keys. The config below uses placeholders and is a local learning setup, not a production-ready deployment. LiteLLM APIs/config fields can change; verify against the official docs linked at the end.

## 1. What is LiteLLM?

LiteLLM provides a unified interface to many model providers and can run as a self-hosted **LLM gateway**.

- **SDK:** call provider models through a common Python interface.
- **Router:** group deployments under a model alias, balance traffic, retry, and fail over.
- **Proxy/Gateway:** expose an OpenAI-compatible HTTP API to applications, centralize authentication, caching, budgets, rate limits, and observability.

A gateway sits between applications and model providers:

`Application / LangChain → LiteLLM Proxy → Router → provider deployments`

The gateway can centralize policy and credentials so every application does not need separate provider-specific integration code.

## 2. Install and set environment variables

For the proxy exercises, run LiteLLM in a terminal. For Python SDK exercises, install the package in the notebook environment.

Set provider keys in your shell or a secrets manager. Do not put real keys into YAML, source control, or notebook output.

In [ ]:
%pip install -U litellm langchain-openai langchain-core pyyaml

In [ ]:
import os

# Example shell setup (run outside Python, or set secrets in your environment):
# export OPENAI_API_KEY="..."
# export ANTHROPIC_API_KEY="..."
# export LITELLM_MASTER_KEY="sk-local-admin-key"
# export LITELLM_SALT_KEY="a-long-stable-random-secret"

# Check only whether variables are present; do not print secret values.
for name in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "LITELLM_MASTER_KEY"]:
    print(name, "configured" if os.getenv(name) else "not configured")

## 3. SDK quick start: one interface

The LiteLLM SDK is useful when you want to call a model directly from Python. Provider model names typically use a provider prefix, such as `openai/...` or `anthropic/...`.

In [ ]:
import os
from litellm import completion

# Requires OPENAI_API_KEY in the environment.
# Replace the model with one available to your account.
# response = completion(
#     model="openai/YOUR_MODEL_NAME",
#     messages=[{"role": "user", "content": "Explain an LLM gateway in two sentences."}],
# )
# print(response.choices[0].message.content)

## 4. Why use the Router?

A `Router` groups one or more deployments behind a stable logical model name. If several deployments share `model_name="chat-model"`, callers request `chat-model` while LiteLLM selects a deployment.

Common benefits:
- load balancing across deployments,
- provider-key and region distribution,
- retry/cooldown behavior,
- fallbacks to another model group,
- optional response caching.

**Important distinction:** multiple deployments can use the same model on separate API keys, regions, or endpoints. Load balancing spreads requests; it does not combine their context windows or make one request run across all keys.

In [ ]:
from litellm import Router

# Illustrative SDK Router configuration. Keep keys in environment variables.
# Replace model names and keys with valid values for your provider.
model_list = [
    {
        "model_name": "chat-model",
        "litellm_params": {
            "model": "openai/YOUR_MODEL_NAME",
            "api_key": os.getenv("OPENAI_API_KEY"),
        },
    },
    # Add another deployment when you have a second valid key / endpoint:
    # {
    #     "model_name": "chat-model",
    #     "litellm_params": {
    #         "model": "openai/YOUR_MODEL_NAME",
    #         "api_key": os.getenv("OPENAI_API_KEY_SECONDARY"),
    #     },
    # },
]

# Uncomment after configuring valid deployments:
# router = Router(model_list=model_list, num_retries=2)
# response = router.completion(
#     model="chat-model",
#     messages=[{"role": "user", "content": "Say hello briefly."}],
# )
# print(response.choices[0].message.content)

## 5. Start the LiteLLM Proxy/Gateway

The Proxy exposes an OpenAI-compatible endpoint, commonly `http://localhost:4000/v1`. Clients authenticate with a LiteLLM key, while provider credentials remain on the gateway.

For production, use a database for managed keys/spend data and Redis for shared caching/rate-limit/router state across multiple proxy replicas. The exact infrastructure depends on the features and deployment mode you enable.

### Example `config.yaml`

This starter configuration defines two logical model groups:
- `fast-chat`: a lower-cost / lower-latency model group for routine requests.
- `strong-chat`: a stronger model group that can be used directly or as a fallback.

Replace model names and provider keys with real deployments available to you. Keep the file local and do not commit secrets.

In [ ]:
# Save the following as config.yaml (copy from this cell into a file).
# LiteLLM supports environment variable references like os.environ/OPENAI_API_KEY.

proxy_config = r'''
model_list:
  - model_name: fast-chat
    litellm_params:
      model: openai/YOUR_FAST_MODEL
      api_key: os.environ/OPENAI_API_KEY
      rpm: 60
      tpm: 60000

  # A second deployment in the same model group can distribute traffic
  # across separate keys or endpoints. Uncomment and configure it if needed.
  # - model_name: fast-chat
  #   litellm_params:
  #     model: openai/YOUR_FAST_MODEL
  #     api_key: os.environ/OPENAI_API_KEY_SECONDARY
  #     rpm: 60
  #     tpm: 60000

  - model_name: strong-chat
    litellm_params:
      model: anthropic/YOUR_STRONG_MODEL
      api_key: os.environ/ANTHROPIC_API_KEY
      rpm: 30
      tpm: 30000

router_settings:
  routing_strategy: simple-shuffle
  num_retries: 2

litellm_settings:
  cache: true
  # For local learning, cache can be in-memory. For shared production cache,
  # configure Redis using the current LiteLLM cache docs.

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY
'''
print(proxy_config)

### Start the gateway

Save the configuration text as `config.yaml`, export the required environment variables, then run the CLI in a terminal:

```bash
litellm --config config.yaml --port 4000
```

The admin/master key is for gateway administration. In real deployments, issue separate virtual keys to applications and users instead of distributing the master key.

## 6. Test the OpenAI-compatible gateway

Once the proxy is running, any compatible client can call its endpoint. The `model` field is the **LiteLLM model alias**, not necessarily the provider's raw model ID.

In [ ]:
from openai import OpenAI

# Requires `pip install -U openai`, a running local proxy, and a key
# matching general_settings.master_key.
# client = OpenAI(
#     api_key=os.environ["LITELLM_MASTER_KEY"],
#     base_url="http://localhost:4000/v1",
# )
# response = client.chat.completions.create(
#     model="fast-chat",
#     messages=[{"role": "user", "content": "Explain caching in one sentence."}],
# )
# print(response.choices[0].message.content)

## 7. Caching: avoid repeated provider calls

Response caching can reduce latency and cost when equivalent requests repeat. A cache hit returns a stored result instead of calling the provider.

Typical considerations:
- **In-memory cache:** easy for local experiments; not shared across replicas and lost on restart.
- **Redis cache:** shared across gateway instances and generally more appropriate for production.
- Cache keys depend on request details; changes to prompts or relevant parameters typically create different entries.
- Avoid caching responses that must be fresh, personalized, or security-sensitive unless the cache behavior is carefully designed.
- Do not confuse **response caching** with provider-side **prompt/prefix caching**; they solve different problems.

In [ ]:
# SDK Router cache example for local experimentation:
# router = Router(model_list=model_list, cache_responses=True)
# response1 = router.completion(
#     model="chat-model",
#     messages=[{"role": "user", "content": "What is 2 + 2?"}],
# )
# response2 = router.completion(
#     model="chat-model",
#     messages=[{"role": "user", "content": "What is 2 + 2?"}],
# )
#
# To use Redis, configure redis_url / Redis settings as documented for your
# installed LiteLLM version. A shared Redis cache is preferred across replicas.

## 8. Smart routing and load balancing

Load balancing selects among deployments that belong to the same logical model group. LiteLLM's Router supports strategies such as shuffle/weighted selection, latency-aware, least-busy, and rate-limit-aware routing (exact availability and settings depend on version).

“Smart routing” can mean two distinct decisions:
1. **Model selection:** choose a small/cheap model for simple requests and a stronger model for harder requests.
2. **Deployment selection:** after a model group is chosen, choose a healthy deployment/key/region within that group.

Start with explicit aliases and routing rules you can observe. Add automatic complexity-based routing only after evaluating quality, latency, and cost.

In [ ]:
# Same model alias, multiple deployments = load balancing.
# Each deployment can use a different API key, region, or endpoint.
#
# model_list = [
#   {"model_name": "fast-chat", "litellm_params": {
#       "model": "openai/YOUR_MODEL", "api_key": os.getenv("OPENAI_API_KEY"),
#       "weight": 1, "rpm": 60, "tpm": 60000}},
#   {"model_name": "fast-chat", "litellm_params": {
#       "model": "openai/YOUR_MODEL", "api_key": os.getenv("OPENAI_API_KEY_SECONDARY"),
#       "weight": 1, "rpm": 60, "tpm": 60000}},
# ]
#
# router = Router(model_list=model_list, routing_strategy="simple-shuffle")
# The alias stays `fast-chat`; callers don't need to know which deployment won.

## 9. Multiple API keys: load balancing is not quota pooling

When the same provider model is configured as multiple deployments with different keys, the router can distribute calls and use another healthy deployment when one is rate limited or unavailable.

Caveats:
- each provider key still has its own provider-side permissions, limits, and billing,
- a key must be authorized for the configured model,
- load balancing does not guarantee perfectly equal usage unless the selected strategy and weights produce it,
- don't bypass a provider's terms or account limits; configure keys you are authorized to use,
- configure deployment RPM/TPM values to reflect realistic provider quotas.

## 10. Fallbacks and retries

**Retry:** attempt the request again, commonly within the same model group/deployment pool.

**Fallback:** move to another configured model group if the primary group fails. For example, a request to `fast-chat` might fall back to `strong-chat` or to a second provider, depending on configuration.

Fallbacks improve availability but can change answer quality, latency, cost, model behavior, and structured-output compatibility. Test fallback paths explicitly.

In [ ]:
# Illustrative fallback configuration for the Proxy YAML:
fallback_config = r'''
router_settings:
  routing_strategy: simple-shuffle
  num_retries: 2
  fallbacks:
    - fast-chat: ["strong-chat"]
'''
print(fallback_config)

The `fallbacks` shape is version/config dependent; confirm the current syntax in the official routing docs before merging this snippet into the main config. Configure fallbacks for retryable provider failures, and avoid silently treating every application-level error as a reason to switch models.

## 11. Rate limiting: two different layers

1. **Provider deployment limits:** RPM/TPM values help the router choose among deployments and avoid overloading them.
2. **Gateway consumer limits:** virtual keys, users, teams, or the server can have their own request/token limits so one application cannot consume all capacity.

In multi-replica production, shared rate-limit state (commonly Redis) matters; in-memory counters on each replica can lead to inconsistent global enforcement. Gateway key/rate-limit features may require the relevant database and configuration.

In [ ]:
# Deployment-level limits are shown in config.yaml:
# litellm_params:
#   model: openai/YOUR_MODEL
#   api_key: os.environ/OPENAI_API_KEY
#   rpm: 60
#   tpm: 60000
#
# For virtual-key RPM/TPM limits, use the current LiteLLM key-management
# API or Admin UI to create a key with limits. Do not guess an admin API
# payload: schemas and required fields can vary by version/deployment.

## 12. Connect LangChain to LiteLLM

The easiest integration is to use LangChain's `ChatOpenAI` with the LiteLLM Proxy's OpenAI-compatible `base_url`. LangChain calls the gateway, and the gateway handles provider translation, routing, caching, retries, fallbacks, and configured rate limits.

This decouples your app from provider-specific model credentials. The `model` passed to LangChain should be a LiteLLM alias configured in the proxy.

In [ ]:
from langchain_openai import ChatOpenAI

# Requires a running LiteLLM Proxy at localhost:4000 and a valid gateway key.
# Use a virtual key in normal application code; master key shown only for local smoke test.
# llm = ChatOpenAI(
#     model="fast-chat",
#     api_key=os.environ["LITELLM_MASTER_KEY"],
#     base_url="http://localhost:4000/v1",
#     temperature=0,
# )
# answer = llm.invoke("Explain why an LLM gateway is useful in two sentences.")
# print(answer.content)

### Add LangChain tools / an agent

You can also use the gateway-backed chat model with LangChain tools or agents. The gateway handles model transport and routing; LangChain still owns your application-level agent loop and tool orchestration.

Tool calling depends on the chosen underlying model and provider path supporting the required tool-call format. Test tool calls on primary and fallback models, because model compatibility may differ.

In [ ]:
# Example pattern; provide your own tool and ensure both primary/fallback models
# support tool calling before enabling this in a real agent.
#
# from langchain_core.tools import tool
# from langchain.agents import create_agent
#
# @tool
# def lookup_status(service: str) -> str:
#     """Return a demo service status."""
#     return f"{service}: operational"
#
# llm = ChatOpenAI(
#     model="fast-chat",
#     api_key=os.environ["LITELLM_MASTER_KEY"],
#     base_url="http://localhost:4000/v1",
# )
# agent = create_agent(model=llm, tools=[lookup_status])
# result = agent.invoke({
#     "messages": [{"role": "user", "content": "Check the API service status."}]
# })
# print(result["messages"][-1].content)

## 13. Observability and cost controls

A gateway can centralize request logs, latency, token usage, spend, and errors. For real deployments:
- use distinct virtual keys for applications or teams,
- set budgets and per-key/team limits where supported,
- add a logging/observability backend,
- track cache-hit rate, fallback rate, provider 429s, latency percentiles, and per-model cost,
- avoid logging sensitive prompts/responses unless your data policy permits it.

## 14. Production architecture notes

For a multi-replica deployment, a common setup is:
- multiple stateless LiteLLM proxy instances behind a load balancer,
- PostgreSQL for key/team/spend/config persistence where required,
- Redis for shared caching and rate-limit/router state,
- a stable `LITELLM_SALT_KEY` for credential encryption where applicable,
- secret management for provider API keys,
- health checks, dashboards, alerting, and a tested upgrade/migration plan.

A single-process local setup is fine for learning but does not provide the same durability or shared coordination.

## 15. Common failure modes

- **Wrong model alias:** the client requests a name not defined in `model_list`.
- **Bad provider prefix:** provider/model format is invalid for the integration.
- **Missing env var:** the proxy starts or request fails because a key is absent.
- **Rate-limit mismatch:** configured RPM/TPM is higher than actual provider quota.
- **Fallback mismatch:** the fallback model does not support the needed tool call, JSON mode, or context length.
- **Cache surprises:** stale or cross-user data is served because cache scope is not appropriate.
- **Master key shared with apps:** excessive privileges; issue scoped virtual keys instead.
- **Multi-replica inconsistency:** cache/rate-limit/router state is local instead of shared.
- **No visibility:** averages hide fallback storms, provider throttling, or a quality regression.

## 16. Revision summary

| Feature | What it does | Main thing to remember |
|---|---|---|
| Unified SDK | Common interface to multiple providers | Provider model names still matter |
| Proxy/Gateway | OpenAI-compatible central endpoint | Clients use gateway credentials and aliases |
| Response cache | Reuses prior responses | Use carefully for freshness/privacy |
| Load balancing | Selects deployment in a model group | Multiple keys are separate deployments |
| Smart model routing | Selects model group by policy/complexity | Evaluate quality/cost trade-offs |
| Retries | Reattempts transient failures | Bound retries to control latency |
| Fallbacks | Switches model group after failure | Fallback model may behave differently |
| Rate limits | Controls provider and consumer usage | Configure both deployment and virtual-key limits |
| LangChain integration | ChatOpenAI points at proxy URL | LangChain handles agent logic; LiteLLM handles gateway routing |

## Suggested hands-on order
1. Start with one model and make one gateway request.
2. Add a second deployment under the same alias and test load balancing.
3. Enable response caching and compare repeated requests.
4. Configure a fallback group and test a controlled failure.
5. Add RPM/TPM and virtual-key limits.
6. Connect `ChatOpenAI` and then a LangChain agent to the proxy.
7. Inspect logs and compare cost/latency/quality.

## Official references

- [LiteLLM documentation](https://docs.litellm.ai/docs/)
- [Proxy / gateway architecture](https://docs.litellm.ai/docs/proxy/architecture)
- [Router and load balancing](https://docs.litellm.ai/docs/routing)
- [Proxy caching](https://docs.litellm.ai/docs/proxy/caching)
- [Budgets and rate limits](https://docs.litellm.ai/docs/proxy/users)
- [LangChain integration](https://docs.litellm.ai/docs/tutorials/langchain)
- [Production deployment](https://docs.litellm.ai/docs/proxy/deploy)

**Note:** Some snippets are intentionally commented out because they require your provider credentials and a running gateway. Treat config fragments as learning templates and validate them against the docs for the LiteLLM version you deploy.